<a href="https://colab.research.google.com/github/purviproduct/personal/blob/main/Purvi_Copy_of_Exercise7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 7: Education remittances
**BDS 5030 | Guided group exercise | October 26, 2026**

Ambler, Aycinena and Yang (2015), *Channeling Remittances to Education*, AEJ: Applied Economics 7(2), 207–232. Paper: https://doi.org/10.1257/app.20140010. Data: https://doi.org/10.3886/E113600V1.

## How we will work
Form a group of 2–3. For each section: **understand the question → view or run the output → discuss and write → review and revise**. Attempt the questions before the TA shows the answers. Keep your initial responses visible and add revisions.

Complete only Q1–Q6 on the answer sheet. Use three materials: this notebook, the classroom slides, and the five-page answer sheet. No separate results booklet is needed. Submit only the handwritten answer sheet, stapled, with all names on page 1, after the final discussion. Everyone contributes and understands the response.

**Open your own working copy:** File → Save a copy in Drive. A group can share one working copy/screen. Use a standard CPU runtime. The data are embedded; no CSV upload, Drive mount, GPU or paid software is needed. Internet is needed for Colab and initial package installation.

Run the three setup cells, then run 7a.1 and 7a.2 with the TA. For 7b, you may read the projected output or run the supplied cells in order. IV is a TA-guided interpretation task. No code writing or editing is required. If setup fails, use a group member’s screen and the projected output.

**AI-free:** No generative AI for writing, explaining, debugging or revising code or answers, including Colab AI. Discuss with your group and ask the TA.

**TA rubric (10 points, six questions):** Q1 descriptive statistics (2), Q2 ITT (2), Q3 selection into use (2), Q4 attrition (2), Q5 household effects (1), Q6 synthesis (1). Relevant attempts, explanation and thoughtful revisions count. Oral discussions and the IV demonstration require no written submission and carry no separate points. Coding proficiency is not graded.

The notebook reproduces selected classroom analyses, not every published table. Keep the supplied statistical code and embedded data unchanged.

## 0. Environment and embedded data
Run the next three setup cells one at a time. Wait for each to finish. Do not use Run all during the guided activity. Pause for group discussion at each section.

One row links one migrant to a nominated target student and household. There are 991 assigned units. Missing spending is not zero.

In [ ]:
# Run this cell first in a fresh CPU runtime; no code editing is required.
# Keep Colab's preloaded scientific libraries intact to avoid a kernel restart.
import sys, subprocess, importlib.metadata as metadata
import numpy as np
import pandas as pd
import scipy
core = {'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__}
required = {'statsmodels': '0.14.5', 'linearmodels': '7.0'}
def installed(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None
if any(installed(p) != v for p, v in required.items()):
    print('Preparing the analysis packages. Please wait; no restart is needed.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           *[f'{p}=={v}' for p, v in {**core, **required}.items()]])
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
# Static HTML table: no interactive widget or external JavaScript is needed.
from IPython.display import HTML, display as notebook_display

def classroom_table(table, decimals=2):
    html = table.to_html(index=False, border=0, escape=True,
                         float_format=lambda x: f'{x:,.{decimals}f}',
                         classes='ex7-classroom-table')
    notebook_display(HTML('<style>\n.ex7-classroom-table {border-collapse:collapse; font-family:Arial,sans-serif; font-size:14px; color:#000; background:#fff; margin:12px 0;}\n.ex7-classroom-table th,.ex7-classroom-table td {padding:10px 16px; text-align:right !important; white-space:nowrap; font-variant-numeric:tabular-nums;}\n.ex7-classroom-table thead th {text-align:center !important;}\n.ex7-classroom-table th:first-child,.ex7-classroom-table td:first-child {text-align:left !important;}\n.ex7-classroom-table thead {border-top:1.5px solid #000; border-bottom:1px solid #000;}\n.ex7-classroom-table tbody tr:last-child {border-bottom:1.5px solid #000;}\n</style>' + '<div style="overflow-x:auto;max-width:100%">' + html + '</div>'))

# Apply the same table layout to all classroom output, including grouped tables.
from html import escape

def baseline_table(table):
    """One baseline table with genuine grouped headers; display copy only."""
    labels = {'us_stu_age': 'Student age',
              'us_stu_female': 'Student female (0/1)',
              'us_mig_yearsofed': 'Migrant schooling (years)'}
    arms = {0: 'Control', 1: 'No match', 2: '1:1 match', 3: '3:1 match'}
    variables = list(dict.fromkeys(table.columns.get_level_values(0)))
    top = ('<tr><th rowspan="2" scope="col">Assigned group</th>'
           '<th rowspan="2" scope="col">Spending outcome</th>')
    for var in variables:
        top += '<th colspan="2" scope="colgroup" class="variable-group">' + escape(labels.get(var, str(var))) + '</th>'
    top += '</tr>'
    second = '<tr>' + ''.join('<th scope="col" class="group-start">N</th><th scope="col">Mean</th>' for _ in variables) + '</tr>'
    rows = []
    previous_arm = None
    for (arm, observed), values in table.iterrows():
        row_class = 'new-arm' if previous_arm is not None and arm != previous_arm else ''
        cells = ['<th scope="row">' + escape(arms.get(arm, str(arm))) + '</th>',
                 '<td class="label">' + ('Observed' if observed == 1 else 'Missing') + '</td>']
        for var in variables:
            count, mean = values[(var, 'count')], values[(var, 'mean')]
            count_text = '—' if pd.isna(count) else f'{int(count):,}'
            mean_text = '—' if pd.isna(mean) else f'{mean:,.4f}'
            cells += ['<td class="group-start">' + count_text + '</td>', '<td>' + mean_text + '</td>']
        rows.append('<tr class="' + row_class + '">' + ''.join(cells) + '</tr>')
        previous_arm = arm
    style = '<style>\n.ex7-baseline {border-collapse:collapse;font-family:Arial,sans-serif;font-size:14px;color:#000;background:#fff;margin:12px 0;border-top:2px solid #000;border-bottom:2px solid #000;}\n.ex7-baseline th,.ex7-baseline td {padding:9px 14px;white-space:nowrap;vertical-align:middle;font-variant-numeric:tabular-nums;}\n.ex7-baseline thead th {text-align:center!important;font-weight:600;}\n.ex7-baseline thead {border-bottom:1px solid #000;}\n.ex7-baseline thead tr:first-child th[colspan] {border-bottom:1px solid #000;}\n.ex7-baseline tbody td {text-align:right!important;}\n.ex7-baseline tbody th,.ex7-baseline tbody .label {text-align:left!important;font-weight:normal;}\n.ex7-baseline .group-start,.ex7-baseline .variable-group {border-left:1px solid #000;}\n.ex7-baseline .new-arm {border-top:1px solid #000;}\n</style>'
    html = (style + '<div style="overflow-x:auto;max-width:100%">'
            '<table class="ex7-baseline"><caption style="text-align:left;font-weight:bold;padding:8px 0">Baseline characteristics by assignment and spending availability</caption>'
            '<thead>' + top + second + '</thead><tbody>' + ''.join(rows) + '</tbody></table></div>'
            '<p style="font-family:Arial;font-size:13px;color:#000">N counts nonmissing baseline values. The female-indicator mean is a proportion, not a percentage.</p>')
    notebook_display(HTML(html))

def display(table):
    if isinstance(table.columns, pd.MultiIndex):
        baseline_table(table)
        return
    view = table.copy()
    # Named indexes contain group information; retain them as ordinary columns.
    if any(name is not None for name in view.index.names):
        view = view.reset_index()
    arms = {0: 'Control', 1: 'No match', 2: '1:1 match', 3: '3:1 match'}
    if 'treatment_match' in view:
        view['treatment_match'] = view['treatment_match'].map(arms)
    if 'outcome_observed' in view:
        view['outcome_observed'] = view['outcome_observed'].map({0: 'Missing', 1: 'Observed'})
    for col in ['missing_rate', 'any_use_rate', 'target_receipt_rate']:
        if col in view:
            view[col] = view[col].map(lambda x: f'{100*x:.2f}%')
    if 'model' in view:
        view['model'] = view['model'].replace({
            'match_a': 'No match vs control', 'match_b': '1:1 vs control',
            'match_c': '3:1 vs control',
            'users_vs_nonusers_noncausal': 'Users vs nonusers',
            'author_continuous_remittance_IV': 'Education spending per $1 of EduRemesa funds (IV)',
            'complete_fu_adultstu_all_available': 'Follow-up completion',
            'stu_cost_imp3_all_available': 'Target spending',
            'hh_cost_imp1_common_sample': 'Household spending',
            'stu_cost_imp3_common_sample': 'Target spending',
            'other_cost_common_sample': 'Other-members spending'})
    view = view.rename(columns={
        'treatment_match': 'Assigned group', 'outcome_observed': 'Spending outcome',
        'assigned': 'Assigned N', 'observed': 'Observed N', 'missing': 'Missing N',
        'missing_rate': 'Missing rate', 'users': 'Users', 'any_use_rate': 'Any-use rate',
        'target_receipt_rate': 'Target receipt rate', 'count': 'Available N',
        'mean': 'Mean', 'model': 'Comparison / outcome', 'clusters': 'Clusters',
        'b': 'Estimate', 'se': 'SE', 'p': 'p-value',
        'low': '95% CI lower', 'high': '95% CI upper'})
    classroom_table(view, decimals=4)

from pathlib import Path
import hashlib
assert all(installed(p) == v for p, v in core.items()), 'Use a fresh runtime.'
print('Python:', sys.version.split()[0])
print({p: installed(p) for p in [*core, *required]})
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('EX7_ENVIRONMENT_READY')


Python: 3.13.16
{'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'statsmodels': '0.14.5', 'linearmodels': '7.0'}
EX7_ENVIRONMENT_READY


In [ ]:
#@title Load the supplied study data (run unchanged)
# The supplied CSV travels with this notebook. Run this cell unchanged.
# Each student's runtime has its own copy; no personal computer/Drive path is used.
import base64, gzip
EMBEDDED_CSV_GZIP = (
    '...'
)
EXPECTED_SHA256 = '1aba0d36027cb6a3c4e742bbffdb7beb1279a7e711558ed70a6df0e4b8363ee3'
csv_bytes = gzip.decompress(base64.b64decode(EMBEDDED_CSV_GZIP))
assert hashlib.sha256(csv_bytes).hexdigest() == EXPECTED_SHA256, 'Embedded data failed validation. Reopen the supplied notebook.'
DATA = Path('ex7_classroom') / 'ex7_data.csv'
DATA.parent.mkdir(exist_ok=True)
if DATA.exists():
    assert hashlib.sha256(DATA.read_bytes()).hexdigest() == EXPECTED_SHA256, 'Existing lesson CSV differs. Start a fresh runtime and reopen the supplied notebook.'
else:
    DATA.write_bytes(csv_bytes)
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == EXPECTED_SHA256
df = pd.read_csv(DATA, dtype={'weekXlocation':str}, float_precision='round_trip')
# Use a clear label for the first column: the migrant-student pair identifier.
df = df.rename(columns={df.columns[0]: 'pair_id'})
assert df.shape == (991, 21) and df.pair_id.is_unique
assert df.outcome_observed.eq(df.stu_cost_imp3.notna().astype(int)).all()
assert df.complete_fu_adultstu.eq(df.outcome_observed).all()
assert df.groupby('treatment_match').size().tolist() == [360,211,203,217]
assert df.er_sent.isin([0,1]).all()
np.testing.assert_array_equal(df[['match_a','match_b','match_c']].to_numpy(),
                              np.column_stack([df.treatment_match.eq(a) for a in [1,2,3]]))
print(f'Data ready: {df.shape[0]} rows, {df.shape[1]} columns; missing outcomes have NOT been removed.')
display(df[['pair_id','treatment_match','stu_cost_imp3','er_sent','outcome_observed']].head())

Data ready: 991 rows, 21 columns; missing outcomes have NOT been removed.


pair_id,Assigned group,stu_cost_imp3,er_sent,Spending outcome
1,Control,NaN,0,Missing
2,Control,0.0000,0,Observed
3,Control,NaN,0,Missing
4,Control,"1,200.0000",0,Observed
5,Control,NaN,0,Missing


In [ ]:
# Match Stata areg/regress cluster corrections and t(G-1) inference.
# Explicit stratum indicators give the same treatment slopes as absorbed FE.
def ols_cluster(data, outcome, regressors='match_c + match_b + match_a', strata=True):
    variables = [outcome, 'dayXlocation'] + regressors.replace(' ', '').split('+')
    if strata:
        variables.append('weekXlocation')
    sample = data.dropna(subset=variables).copy()
    formula = outcome + ' ~ ' + regressors + (' + C(weekXlocation)' if strata else '')
    model = smf.ols(formula, data=sample)
    assert np.linalg.matrix_rank(model.exog) == model.exog.shape[1]
    fit = model.fit(cov_type='cluster', cov_kwds={'groups':sample.dayXlocation,
                    'use_correction':True, 'df_correction':True}, use_t=True)
    return fit, sample

def row(fit, sample, term, model):
    ci = fit.conf_int().loc[term]
    return dict(model=model, N=len(sample), clusters=sample.dayXlocation.nunique(),
                b=fit.params[term], se=fit.bse[term], p=fit.pvalues[term],
                low=ci.iloc[0], high=ci.iloc[1])

results = []

## Brief case introduction

Listen to the TA; no written response is required here. Salvadoran migrants were randomly assigned to control, an EduRemesa offer without matching funds, a 1:1 match, or a 3:1 match. All groups received education information.

One row links a migrant to a nominated target student and household. `treatment_match` is assignment, `er_sent` is actual product use, and `stu_cost_imp3` is target-student education spending in US dollars. The file retains all 991 assigned units.

**The first written task is 7a.1: descriptive statistics, on answer-sheet page 3.** The variable guide is on answer-sheet page 2. The row identifier `pair_id` identifies one migrant–student pair.


## 7a.1 · Q1 — answer sheet p. 3
Run the next cell. Record only the control and 3:1 means and their difference. Explain why missing spending should not become zero. Arm codes: 0 = control, 1 = no match, 2 = 1:1, 3 = 3:1. Quartiles, SD and SE are discussed orally; no additional entries are required.


In [ ]:
# Stata's default sample-percentile definition is NumPy averaged_inverted_cdf,
# NOT pandas' default linear interpolation. SD uses n-1.
records = []
for arm, group in df.groupby('treatment_match'):
    y = group.stu_cost_imp3.dropna()
    p25, median, p75 = np.quantile(y, [.25,.5,.75], method='averaged_inverted_cdf')
    records.append(dict(arm=arm, assigned=len(group), observed=len(y), mean=y.mean(),
                        median=median, p25=p25, p75=p75, sd=y.std(ddof=1)))
descriptives = pd.DataFrame(records)
# Format a display copy only; keep full-precision analysis data unchanged.
descriptive_view = descriptives.copy()
descriptive_view['arm'] = descriptive_view['arm'].map(
    {0: 'Control', 1: 'No match', 2: '1:1 match', 3: '3:1 match'})
descriptive_view = descriptive_view.rename(columns={
    'arm': 'Assigned group', 'assigned': 'Assigned N', 'observed': 'Observed N',
    'mean': 'Mean ($)', 'median': 'Median ($)', 'p25': 'P25 ($)',
    'p75': 'P75 ($)', 'sd': 'SD ($)'})
classroom_table(descriptive_view)
raw_difference = descriptives.set_index('arm').loc[3,'mean'] - descriptives.set_index('arm').loc[0,'mean']
print(f'Unadjusted mean difference (3:1 minus control): ${raw_difference:,.2f}')

Assigned group,Assigned N,Observed N,Mean ($),Median ($),P25 ($),P75 ($),SD ($)
Control,360,270,"1,358.07","1,214.00",0.00,"2,075.00","1,281.27"
No match,211,153,"1,325.64","1,350.00",265.00,"1,975.00","1,063.64"
1:1 match,203,150,"1,405.29","1,306.00",0.00,"2,056.00","1,272.22"
3:1 match,217,155,"1,624.91","1,475.00",578.00,"2,395.00","1,370.39"


Unadjusted mean difference (3:1 minus control): $266.84


**Pause:** Complete only the indicated numbered question. Review and revise after discussion.


## 7a.2 · Q2 — Intention-to-Treat - answer sheet p. 3
After calculating the raw gap, discuss orally what the 3:1 students would have spent under control assignment and why randomized control helps approximate that counterfactual. No separate written response is required.

Run the next cell with the TA. Record only the 3:1 estimate and 95% CI, then interpret the assignment comparison in one sentence. `match_c` identifies 3:1. The model uses week–location strata and day–location clustered uncertainty. Nonusers, missingness and the direct 3:1 versus 1:1 test are oral follow-ups.


In [ ]:
itt, itt_sample = ols_cluster(df, 'stu_cost_imp3')
results.append(row(itt, itt_sample, 'match_c', 'stu_cost_imp3_all_available'))
display(pd.DataFrame([row(itt,itt_sample,t,t) for t in ['match_a','match_b','match_c']]))

Comparison / outcome,N,Clusters,Estimate,SE,p-value,95% CI lower,95% CI upper
No match vs control,728,125,19.3230,111.5221,0.8627,-201.4105,240.0565
1:1 vs control,728,125,74.9746,117.0269,0.5229,-156.6545,306.6037
3:1 vs control,728,125,301.4924,125.4658,0.0177,53.1605,549.8244


**Pause:** Complete only the indicated numbered question. Review and revise after discussion.


## 7b.1 · Q3 — Selection Bias - answer sheet p. 4
View the output or run the next cell. In one or two sentences, explain why a users–nonusers spending difference is not automatically causal. A source of selection can help explain your reasoning. Rates use all assigned units. The distinction between any use and target-student receipt is discussed orally; no rates need to be copied.


In [ ]:
takeup = df.groupby('treatment_match').agg(assigned=('pair_id','size'),
    users=('er_sent','sum'), any_use_rate=('er_sent','mean'),
    target_receipt_rate=('er_target_student','mean'))
display(takeup)
naive, naive_sample = ols_cluster(df, 'stu_cost_imp3', 'er_sent', strata=False)
results.append(row(naive, naive_sample, 'er_sent', 'users_vs_nonusers_noncausal'))
display(pd.DataFrame([results[-1]]))

Assigned group,Assigned N,Users,Any-use rate,Target receipt rate
Control,360,0,0.00%,0.00%
No match,211,0,0.00%,0.00%
1:1 match,203,10,4.93%,4.43%
3:1 match,217,31,14.29%,11.98%


Comparison / outcome,N,Clusters,Estimate,SE,p-value,95% CI lower,95% CI upper
Users vs nonusers,728,125,270.7024,194.5630,0.1666,-114.3922,655.7970


**Pause:** Complete only the indicated numbered question. Review and revise after discussion.


## 7b.2 · Education spending per $1 of EduRemesa funds (IV)
**TA-guided demonstration. No written response.** Read the projected output; running the next cell is optional.

Compare control and 3:1 only. Randomized assignment to 3:1 (`match_c`) instruments total EduRemesa funds for the target student (`er_ts_amount_total`). **Funds include migrant contributions plus matching subsidies.** The outcome is target-student education spending (`stu_cost_imp3`).

The estimate measures dollars of education spending per additional $1 of EduRemesa funds induced by assignment. Discuss units and the exclusion assumption orally. It is not a return per subsidy dollar or a binary-use LATE. No additional written answer is required.

In [ ]:
iv_sample = df.loc[df.match_a.eq(0) & df.match_b.eq(0)].dropna(
    subset=['stu_cost_imp3','er_ts_amount_total','match_c','weekXlocation','dayXlocation']).copy()
# Stata ivregress without small uses an un-debiased cluster covariance and normal inference.
iv = IV2SLS.from_formula('stu_cost_imp3 ~ 1 + C(weekXlocation) + [er_ts_amount_total ~ match_c]',
                         iv_sample).fit(cov_type='clustered', clusters=iv_sample.dayXlocation,
                                        debiased=False)
term = 'er_ts_amount_total'
ci = iv.conf_int().loc[term]
results.append(dict(model='author_continuous_remittance_IV',N=len(iv_sample),
    clusters=iv_sample.dayXlocation.nunique(), b=iv.params[term], se=iv.std_errors[term],
    p=iv.pvalues[term], low=ci.iloc[0], high=ci.iloc[1]))
display(pd.DataFrame([results[-1]]))
# Report the small-sample-corrected first-stage F used by Stata estat firststage.
# linearmodels' default first-stage diagnostic is a different Wald chi-square convention.
first, first_sample = ols_cluster(iv_sample, 'er_ts_amount_total', 'match_c')
first_stage_F = float(first.f_test('match_c = 0').fvalue)

Comparison / outcome,N,Clusters,Estimate,SE,p-value,95% CI lower,95% CI upper
Education spending per $1 of EduRemesa funds (IV),425,117,3.7201,1.6472,0.0239,0.4916,6.9485


**Pause for oral discussion:** No written answer is required for IV.


## 7b.3 · Q4 — Attrition - answer sheet p. 4
View the output or run the next cell. Compare follow-up completion for all three treatment groups with control, then inspect the joint F-test that all three assignment coefficients equal zero. Coefficients and confidence intervals below are in percentage points.

For Q4, continue to focus on the 3:1 versus control comparison provided on the answer sheet. Explain why a nonsignificant difference does not establish absence of attrition bias. No table copying or additional written response is required. Baseline summaries and the joint test are TA-guided diagnostics.

In [ ]:
df['attrition'] = 1 - df.outcome_observed
print('Assigned units and spending-outcome availability:')
display(df.groupby('treatment_match').agg(assigned=('pair_id','size'),
    observed=('outcome_observed','sum'), missing=('attrition','sum'), missing_rate=('attrition','mean')))

# Same completion regression, with all three assignment effects displayed.
response, response_sample = ols_cluster(df, 'complete_fu_adultstu')
results.append(row(response, response_sample, 'match_c', 'complete_fu_adultstu_all_available'))
response_rows = pd.DataFrame([
    row(response, response_sample, term, term)
    for term in ['match_a', 'match_b', 'match_c']
])
# Convert a display copy to percentage points; keep model estimates unchanged.
response_view = response_rows.copy()
response_view[['b','se','low','high']] *= 100
response_view = response_view.rename(columns={
    'b':'Estimate (pp)', 'se':'SE (pp)',
    'low':'95% CI lower (pp)', 'high':'95% CI upper (pp)'
})
print('Effects on follow-up completion: each assigned group versus control.')
display(response_view)

joint_completion = response.f_test('match_a = 0, match_b = 0, match_c = 0')
print('Joint test: all three assignment effects on completion equal zero.')
print(f'Cluster-robust F({joint_completion.df_num:.0f}, {joint_completion.df_denom:.0f}) = '
      f'{float(joint_completion.fvalue):.4f}; p-value = {float(joint_completion.pvalue):.4f}')
print('Nonsignificance does not establish absence of attrition bias: similar rates can hide different selection.')

print('Baseline characteristics by assignment and spending availability:')
display(df.groupby(['treatment_match','outcome_observed'])[
    ['us_stu_age','us_stu_female','us_mig_yearsofed']].agg(['count','mean']))

Assigned units and spending-outcome availability:


Assigned group,Assigned N,Observed N,Missing N,Missing rate
Control,360,270,90,25.00%
No match,211,153,58,27.49%
1:1 match,203,150,53,26.11%
3:1 match,217,155,62,28.57%


Effects on follow-up completion: each assigned group versus control.


Comparison / outcome,N,Clusters,Estimate (pp),SE (pp),p-value,95% CI lower (pp),95% CI upper (pp)
No match vs control,991,126,-2.6605,3.7438,0.4786,-10.0700,4.7490
1:1 vs control,991,126,-2.3986,3.6321,0.5102,-9.5870,4.7899
3:1 vs control,991,126,-3.4544,3.5494,0.3323,-10.4791,3.5704


Joint test: all three assignment effects on completion equal zero.
Cluster-robust F(3, 125) = 0.4563; p-value = 0.7133
Nonsignificance does not establish absence of attrition bias: similar rates can hide different selection.
Baseline characteristics by assignment and spending availability:


**Pause:** Complete only the indicated numbered question. Review and revise after discussion.


## 7b.4 · Q5 — Spillover Effect - answer sheet p. 5
View the common-sample output or run the next cell. Use the supplied other-members confidence interval to explain whether the result establishes no effect. Common-sample reasoning and the limits of spillover interpretation are discussed orally. No coefficient table needs to be filled.


In [ ]:
common = df.dropna(subset=['other_cost','hh_cost_imp1','stu_cost_imp3']).copy()
# other_cost is the stored dataset variable; do not silently redefine it.
np.testing.assert_allclose(common.other_cost,common.hh_cost_imp1-common.stu_cost_imp3,
                           rtol=1e-6, atol=0.001)
household_rows = []
for y in ['hh_cost_imp1','stu_cost_imp3','other_cost']:
    fit, sample = ols_cluster(common, y)
    entry = row(fit, sample, 'match_c', y+'_common_sample')
    household_rows.append(entry)
    results.append(entry)
display(pd.DataFrame(household_rows))

Comparison / outcome,N,Clusters,Estimate,SE,p-value,95% CI lower,95% CI upper
Household spending,728,125,332.7570,168.7065,0.0508,-1.1605,666.6745
Target spending,728,125,301.4924,125.4658,0.0177,53.1605,549.8244
Other-members spending,728,125,31.2646,101.7034,0.7590,-170.0349,232.5641


**Pause:** Complete only the indicated numbered question. Review and revise after discussion.


## Overall judgment · Q6 — answer sheet p. 5
Before the example appears, write two sentences: the study’s most convincing finding and one important limitation. Review and revise after discussion. Submit one completed five-page answer sheet per group, with all names, after the final discussion. No notebook or output-file submission is required. The final code cell is an optional run check for groups that ran every analysis section.


**Pause:** Complete only the indicated numbered question. Review and revise after discussion.


In [ ]:
model_results = pd.DataFrame(results)
model_results.to_csv('python_classroom_results.csv', index=False)
descriptives.to_csv('python_descriptive_statistics.csv', index=False)
print('EX7_COLAB_RUN_COMPLETE')
print('Review and revise your group answer sheet; submit it after the final discussion.')


EX7_COLAB_RUN_COMPLETE
Review and revise your group answer sheet; submit it after the final discussion.
